# Module 03 — ML for Forecasting

## Notebook 4 · Recursive vs Direct vs Purely Causal Forecasting

**Goals**

1. Understand the **bias-variance trade-off** between recursive and direct multi-horizon forecasting.
2. Train a **DirectMultiHorizonForecaster**: one LightGBM model per horizon h = 1..H.
3. Train a **PurelyCausalForecaster**: same as direct, but with a feature whitelist that drops anything that wouldn't be available at production time.
4. Compare the three strategies on the same holdout.

| Strategy | # models | Error compounding | Production-ready? |
|---|---|---|---|
| Recursive | 1 | Yes — predictions feed lag features | If features are causal |
| Direct | H | No | If features are causal |
| Purely causal direct | H | No | **Yes by construction** |


In [1]:
# === Colab / local setup ====================================================
# 1. Install dependencies (uncomment the pip line on first Colab run).
# !pip install -q lightgbm==4.* prophet plotly optuna shap pandas numpy scikit-learn pyarrow

# 2. Make the `utils` package importable. Two options:
#    (a) Notebook is sitting next to a `utils/` folder (recommended).
#    (b) The package is uploaded as a zip; unzip it and `sys.path.append(...)`.
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__"))  # may be empty in Colab
for cand in [".", "..", "/content", "/content/ml_forecasting_tutorial"]:
    if os.path.isdir(os.path.join(cand, "utils")):
        sys.path.insert(0, cand)
        break

# 3. Standard imports for every notebook.
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "colab"   # works in Colab + Jupyter

# 4. Tell pandas to display nicely.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [2]:
# === Dataset configuration ==================================================
# EDIT THIS CELL TO POINT AT YOUR DATASET
DATA_PATH    = "./dataset/m5/m5_small.csv"             # path to your CSV / parquet
DATE_COL     = "date"                              # date column
TARGET_COL   = "sales"                          # target / forecast column
KEY_COLS     = ['item_id',
                'dept_id',
                'cat_id',
                'store_id',
                'state_id']                           # columns identifying a unique series
EXCLUDE_COLS = []
FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
HOLDOUT_DAYS = 28                               # length of test horizon


In [3]:
from utils.data_utils import load_forecasting_data, time_based_split
from utils.feature_engineering import FeatureEngineer
from utils.lgbm_forecaster import (
    RecursiveForecaster, DirectMultiHorizonForecaster, PurelyCausalForecaster,
)

df = load_forecasting_data(DATA_PATH, DATE_COL, TARGET_COL, KEY_COLS, exclude_cols=EXCLUDE_COLS)
cutoff = df[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train_df, test_df = time_based_split(df, DATE_COL, cutoff)

val_cutoff = train_df[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
fit_df, val_df = time_based_split(train_df, DATE_COL, val_cutoff)

fe = FeatureEngineer(
    date_col=DATE_COL, target_col=TARGET_COL, key_cols=KEY_COLS,
    lags=[1, 2, 7, 14, 28],
    rolling_windows=[7, 14, 28],
    rolling_stats=["mean", "std", "max"],
    ewm_halflives=[7.0, 28.0],
)

base_params = {
    "objective": "tweedie", "tweedie_variance_power": 1.2,
    "metric": "rmse", "learning_rate": 0.05,
    "num_leaves": 63, "min_data_in_leaf": 50,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
    "lambda_l2": 0.1, "verbose": -1, "verbose_eval": 10,
}


### 1. Recursive — same model as Notebook 3


In [5]:
recursive = RecursiveForecaster(fe, params=base_params, num_boost_round=2000, early_stopping_rounds=100)
recursive.fit(fit_df, valid_df=val_df)
print(f"recursive  best_iter = {recursive.model.best_iteration}")


recursive  best_iter = 890


### 2. Direct multi-horizon

We train **one model per horizon h ∈ {1, 2, ..., H}**. The h-th model learns a function `(features at time t) → y_{t+h}`. There is no recursion at inference: from the anchor date we directly produce H predictions in one shot.

**Cost note.** Training H boosters is roughly H× slower than recursive. For pedagogical purposes we use a smaller `num_boost_round` and a coarse horizon set; in production you would parallelise across horizons or keep H small (e.g., {1, 7, 14, 28} for weekly milestones).


In [6]:
HORIZONS = [1, 7, 14, 21, 28]   # subset of horizons to keep training quick

direct = DirectMultiHorizonForecaster(
    feature_engineer=fe,
    horizons=HORIZONS,
    params=base_params,
    num_boost_round=800,
    early_stopping_rounds=80,
)
direct.fit(fit_df, valid_df=val_df, verbose=True)


  horizon h= 1  trained, best_iter=641
  horizon h= 7  trained, best_iter=752
  horizon h=14  trained, best_iter=435
  horizon h=21  trained, best_iter=689
  horizon h=28  trained, best_iter=800


### 3. Purely causal direct

Same as Direct, but the feature list is filtered down to columns that are guaranteed to be available at production time: lags, rolling-window stats over lagged targets, calendar features, and keys. We drop any column whose name doesn't match the causal whitelist.

In real projects you would extend `extra_causal=` with the names of any additional regressors you have committed in advance (planned promotion calendar, holiday calendar, etc.).


In [6]:
causal = PurelyCausalForecaster(
    feature_engineer=fe,
    horizons=HORIZONS,
    params=base_params,
    num_boost_round=800,
    early_stopping_rounds=80,
    extra_causal=[],
)
causal.fit_causal(fit_df, valid_df=val_df, verbose=True)
print(f"  direct features = {len(direct.feature_names)}")
print(f"  causal features = {len(causal.feature_names)}")
print(f"  dropped         = {set(direct.feature_names) - set(causal.feature_names)}")


  causal h= 1  trained
  causal h= 7  trained
  causal h=14  trained
  causal h=21  trained
  causal h=28  trained
  direct features = 81
  causal features = 38
  dropped         = {'event_name_1_VeteransDay', 'event_name_1_NBAFinalsStart', 'snap_TX', 'event_name_1_NewYear', 'event_type_1_Sporting', 'event_name_1_LaborDay', 'snap_WI', 'event_name_1_IndependenceDay', 'event_type_2_Religious', 'event_name_1_Ramadan starts', 'event_name_1_Easter', 'event_name_1_LentWeek2', 'event_name_1_MartinLutherKingDay', 'event_name_2_Cinco De Mayo', 'event_name_1_Halloween', "event_name_1_Father's day", 'event_name_1_OrthodoxEaster', 'event_name_1_Thanksgiving', 'event_name_1_Cinco De Mayo', 'event_name_1_Christmas', 'event_name_1_EidAlAdha', 'event_type_1_Religious', 'event_name_1_Chanukah End', 'event_name_1_PresidentsDay', "event_name_2_Father's day", 'event_name_1_ValentinesDay', 'event_name_1_NBAFinalsEnd', 'event_name_1_Eid al-Fitr', 'event_name_1_Pesach End', 'event_name_1_SuperBowl', 'event_t

### 4. Generate test-horizon predictions for each strategy

The Recursive model uses iterative roll-out. The Direct / Causal models use the **anchor date** = the last day of training, and produce one prediction per `h ∈ HORIZONS`. We will compare results only on the dates each model can produce.


In [ ]:
horizon_dates = pd.date_range(test_df[DATE_COL].min(), test_df[DATE_COL].max(), freq=FREQ)
# Dataset Specific features
holiday_cols = [c for c in train_df.columns if c.startswith("event_")]
snap_cols = [c for c in train_df.columns if c.startswith("snap_")]
other_future_known_cols = ['sell_price']
exo_future = test_df[[DATE_COL] + KEY_COLS + holiday_cols + snap_cols + other_future_known_cols].copy()

preds_recursive = recursive.predict_recursive(history_df=train_df, horizon_dates=horizon_dates, exogenous_future=exo_future,)
preds_direct    = direct.predict(history_df=train_df, anchor_date=train_df[DATE_COL].max())
preds_causal    = causal.predict(history_df=train_df, anchor_date=train_df[DATE_COL].max())

# Direct/causal only produce predictions for the configured horizons; subset
# the actual test set to the matching dates for a fair comparison.
target_dates = preds_direct[DATE_COL].unique()
mask = test_df[DATE_COL].isin(target_dates)
test_subset = test_df.loc[mask].copy()

print(f"  comparison dates : {len(target_dates)}  ({sorted(target_dates)})")


### 5. Evaluate all three on the same dates


In [ ]:
from utils.metrics import metric_report

def join(actual, predicted):
    return actual.merge(predicted, on=[DATE_COL, *KEY_COLS], how="inner")

j_recursive = join(test_subset[[DATE_COL, *KEY_COLS, TARGET_COL]], preds_recursive)
j_direct    = join(test_subset[[DATE_COL, *KEY_COLS, TARGET_COL]], preds_direct[[DATE_COL, *KEY_COLS, "yhat"]])
j_causal    = join(test_subset[[DATE_COL, *KEY_COLS, TARGET_COL]], preds_causal[[DATE_COL, *KEY_COLS, "yhat"]])

table = pd.DataFrame({
    "Recursive":   metric_report(j_recursive[TARGET_COL], j_recursive["yhat"]),
    "Direct":      metric_report(j_direct[TARGET_COL],    j_direct["yhat"]),
    "PurelyCausal":metric_report(j_causal[TARGET_COL],    j_causal["yhat"]),
}).round(3)
table


### 6. Visualise the three strategies on a single key


In [ ]:
from utils.viz import plot_forecast_grid

key = test_df[KEY_COLS].drop_duplicates().iloc[0].tolist()
def select(df_in, key_values):
    out = df_in.copy()
    for c, v in zip(KEY_COLS, key_values):
        out = out[out[c] == v]
    return out

hist = select(train_df, key).tail(120)
act  = select(test_subset, key)

fig = plot_forecast_grid(
    forecasts={
        "Recursive":     select(preds_recursive, key)[[DATE_COL, "yhat"]],
        "Direct":        select(preds_direct,    key)[[DATE_COL, "yhat"]],
        "Purely causal": select(preds_causal,    key)[[DATE_COL, "yhat"]],
    },
    actual=act, date_col=DATE_COL, target_col=TARGET_COL, history=hist,
    title=f"Strategy comparison — {dict(zip(KEY_COLS, key))}",
)
fig.show()


### 7. Error growth by horizon

Recursive forecasting *should* show the classic error-growth pattern: errors get worse the further ahead you predict because each step's prediction error feeds the next step's lag features. Direct forecasting decouples each horizon and tends to be flatter.


In [ ]:
import plotly.graph_objects as go
from utils.viz import _apply_theme, PALETTE

def per_horizon_metric(joined_df, anchor_date):
    j = joined_df.copy()
    j["horizon"] = (j[DATE_COL] - pd.Timestamp(anchor_date)).dt.days
    return (j.groupby("horizon")
              .apply(lambda g: float(((g[TARGET_COL] - g["yhat"]).abs()).sum() / max(g[TARGET_COL].abs().sum(), 1) * 100))
              .rename("WAPE%").reset_index())

anchor = train_df[DATE_COL].max()
m_rec = per_horizon_metric(j_recursive, anchor)
m_dir = per_horizon_metric(j_direct,    anchor)
m_cau = per_horizon_metric(j_causal,    anchor)

fig = go.Figure()
fig.add_trace(go.Scatter(x=m_rec["horizon"], y=m_rec["WAPE%"], mode="lines+markers", name="Recursive"))
fig.add_trace(go.Scatter(x=m_dir["horizon"], y=m_dir["WAPE%"], mode="lines+markers", name="Direct"))
fig.add_trace(go.Scatter(x=m_cau["horizon"], y=m_cau["WAPE%"], mode="lines+markers", name="Purely causal"))
_apply_theme(fig, title="WAPE% by horizon (days ahead)", height=400,
             xaxis_title="horizon (days ahead)", yaxis_title="WAPE%").show()


### Recap

* **Recursive** is cheap to train, slow to predict, and accumulates error.
* **Direct** is expensive to train, fast to predict, and avoids compounding — but each model has a smaller usable feature set the further out you go.
* **Purely causal direct** is the production-grade default: explicit feature whitelisting is the only way to guarantee no leakage at deploy time.

**Next.** Notebook 5 — hyper-parameter optimisation with Grid Search and Optuna.
